# ColorMNIST Simulation Results

Every table below comes from the multi-seed ColorMNIST runs in
`Simulations/ColorMNIST/dr_mmd_flow/results`. The **default configuration** is

- projected Nystrom solver,
- **no-split** nuisance fitting (no cross-fitting),
- **no-split** autoencoder (`ae_split = none`),
- initialisation from the observed arm (`init = observed`),
- `M = 2000` particles, `n_obs = 20000` logged observations,
- the design `w = 10`, `tau = 0.08`, `alpha = 0.2`, `k_mask = 20`, digits `(1, 6)`,
- 3 data seeds, both intervention arms, 1 repeat per cell.

**These are the corrected design.** Earlier runs were generated with `alpha = 0.0` and
`k_mask = 10` — not the recolouring the DecFM reference numbers in Section 1 were produced on.
Those runs are archived under `results_archive_2026-09-29_alpha0.0_k10` and are not loaded
here; every CSV in `results/` carries `alpha = 0.2`, `k_mask = 20`, `tau = 0.08`, and
`analysis_tables.load` refuses to build a table from a mixed design. Coverage is complete: 138
result rows across 23 configuration cells, every cell at the full 6 rows (3 seeds x 2 arms).

Section 1 reports that configuration against the published DecFM numbers. **Every later
section changes exactly one thing** and leaves the rest at the default. All numbers are
`mean ± sd`, where the arms are averaged within a run and the spread is the standard
deviation across the 3 data seeds.

In [1]:
import pandas as pd

import analysis_tables as at

runs = at.load("ColorMNIST")
runs["tag"] = runs["tag"].astype(str)

# The default configuration, and the three sweeps that branch off it.
sweep = runs[runs.tag == "sweep"]
wsweep = runs[runs.tag == "wsweep"]

print("rows:", len(runs), " seeds:", sorted(runs.data_seed.unique()),
      " arms:", sorted(runs.target_a.unique()), " repeats:", sorted(runs.repeat.unique()))

rows: 138  seeds: [np.int64(0), np.int64(1), np.int64(2)]  arms: [np.int64(0), np.int64(1)]  repeats: [np.int64(0)]


In [2]:
# Coverage: how many rows sit in each cell of the grid (3 seeds x 2 arms = 6 is complete).
coverage = (runs.groupby(["tag", "init", "dr_fit", "n_obs", "n_particles_requested", "w_conf"])
                .size().rename("rows").reset_index())
print(coverage.to_string(index=False))
print()
print("cells:", len(coverage), " | all complete:", bool((coverage.rows == 6).all()))

   tag     init   dr_fit  n_obs  n_particles_requested  w_conf  rows
 sweep   moment  nosplit  20000                   2000    10.0     6
 sweep   normal  nosplit  20000                   2000    10.0     6
 sweep observed crossfit   5000                   2000    10.0     6
 sweep observed crossfit  10000                   2000    10.0     6
 sweep observed crossfit  15000                   2000    10.0     6
 sweep observed crossfit  20000                   2000    10.0     6
 sweep observed  nosplit   5000                   2000    10.0     6
 sweep observed  nosplit  10000                   2000    10.0     6
 sweep observed  nosplit  15000                   2000    10.0     6
 sweep observed  nosplit  20000                    250    10.0     6
 sweep observed  nosplit  20000                    500    10.0     6
 sweep observed  nosplit  20000                   1000    10.0     6
 sweep observed  nosplit  20000                   2000    10.0     6
wsweep observed  nosplit  20000   

## 1. Headline

The default configuration against the three DecFM baselines of Dance et al., reproduced
verbatim. Same dataset, same metrics, same lower-is-better convention.

In [3]:
print(at.headline_table(runs, "ColorMNIST", at.DECFM_CMNIST))

ColorMNIST   (n_obs = 20000, M = 2000, seeds [0, 1, 2])

Method                SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-----------------------------------------------------------------------------------------------
DecFM-Plugin          0.01137 ± 0.00078    32.005 ± 1.075       0.08206 ± 0.00754    0.01991 ± 0.00285
DecFM-DR-I            0.01392 ± 0.00090    33.650 ± 1.151       0.09217 ± 0.00657    0.03665 ± 0.01237
DecFM-DR-OT           0.01190 ± 0.00106    33.824 ± 1.219       0.07794 ± 0.00384    0.02840 ± 0.00866
DR MMD Flow (ours)    0.00939 ± 0.00044    30.618 ± 0.886       0.08016 ± 0.00107    0.02053 ± 0.00357

Ours: projected Nystrom, no-split nuisance, no-split autoencoder, P_Y initialisation.
FID is the legacy TorchVision Inception-v3 scorer, matching the reference convention.


In [4]:
# What the flow started from, and the two limits it is working between.
default = at.default_rows(runs)
rows = []
for label, suffix in [("No flow (logged arm)", "initial"),
                      ("DR MMD Flow (ours)", "flow"),
                      ("Autoencoder round-trip", "roundtrip"),
                      ("Oracle floor", "floor")]:
    columns = ["sw2", "fid_legacy", "latent_sw2", "colour_w2"]
    rows.append((label, [at.summarise(default, f"{c}_{suffix}")
                         if f"{c}_{suffix}" in default.columns else None for c in columns]))

print(at.plain_table(rows, ["SW2", "FID", "Latent SW2", "Color-W2"],
                     first_column="Row", label_width=24,
                     title="ColorMNIST   default configuration, the reference rows",
                     note="Round-trip = the oracle sample pushed through the autoencoder, "
                          "i.e. the best any sampler in this latent space could score.\n"
                          "Oracle floor = a second independent oracle draw; latent SW2 has "
                          "no round-trip row (it is measured before decoding)."))

ColorMNIST   default configuration, the reference rows

Row                     SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-------------------------------------------------------------------------------------------------
No flow (logged arm)    0.04455 ± 0.00006    43.059 ± 0.719       0.50118 ± 0.01925    0.25171 ± 0.00263
DR MMD Flow (ours)      0.00939 ± 0.00044    30.618 ± 0.886       0.08016 ± 0.00107    0.02053 ± 0.00357
Autoencoder round-trip  0.00818 ± 0.00074    30.576 ± 0.761       --                   0.01416 ± 0.00203
Oracle floor            0.00635 ± 0.00025    3.343 ± 0.102        0.07178 ± 0.00418    0.00674 ± 0.00099

Round-trip = the oracle sample pushed through the autoencoder, i.e. the best any sampler in this latent space could score.
Oracle floor = a second independent oracle draw; latent SW2 has no round-trip row (it is measured before decoding).


We are best on SW2 (0.00939 vs 0.01137 for the strongest DecFM baseline, about 17% below it) and best on
FID (30.62 vs 32.0–33.8), though the FID margin is only slightly wider than the seed spread. Latent SW2
is a tie (0.08016 against 0.07794–0.09217) and Color-W2 (0.02053) ties the plug-in (0.01991) while beating
both DR baselines (0.02840–0.03665). The second table says why the FID column should not be over-read: the
autoencoder round-trip already costs 30.58, so our 30.62 is within noise of the ceiling this 28-pixel
decoder imposes, while the raw oracle floor is 3.34.

## 2. Ablation: nuisance fitting

No-split (the default) against cross-fitted nuisances. Everything else at the default:
`n_obs = 20000`, `M = 2000`, observed initialisation.

In [5]:
nuisance = sweep[(sweep.init == "observed") & (sweep.n_obs == 20000)
                 & (sweep.n_particles_requested == 2000)]

print(at.ablation_table(
    nuisance, "ColorMNIST", "dr_fit",
    {"nosplit": "No-split (default)", "crossfit": "Cross-fitted"},
    order=["nosplit", "crossfit"], first_column="Nuisance fit",
    title="ColorMNIST   nuisance fitting   (n_obs = 20000, M = 2000, 3 seeds)"))

ColorMNIST   nuisance fitting   (n_obs = 20000, M = 2000, 3 seeds)

Nuisance fit          SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-----------------------------------------------------------------------------------------------
No-split (default)    0.00939 ± 0.00044    30.618 ± 0.886       0.08016 ± 0.00107    0.02053 ± 0.00357
Cross-fitted          0.00942 ± 0.00016    30.736 ± 0.986       0.08054 ± 0.00132    0.02063 ± 0.00340


No detectable difference on any metric: every cross-fitted value is inside the no-split
standard deviation (SW2 0.00942 vs 0.00939, FID 30.74 vs 30.62). Cross-fitting costs a factor
of `n_folds` in nuisance compute and buys nothing here, which is why the no-split run is the
default.

## 3. Ablation: number of observations

The `n_obs` sweep at the default particle count and initialisation. Both nuisance arms were
run across the full sweep, so both are shown.

In [6]:
by_n = sweep[(sweep.init == "observed") & (sweep.n_particles_requested == 2000)]

print(at.ablation_table(
    by_n[by_n.dr_fit == "nosplit"], "ColorMNIST", "n_obs", {},
    first_column="n_obs", label_width=14,
    title="ColorMNIST   sample size, no-split nuisance (default)   (M = 2000, 3 seeds)"))
print()
print(at.ablation_table(
    by_n[by_n.dr_fit == "crossfit"], "ColorMNIST", "n_obs", {},
    first_column="n_obs", label_width=14,
    title="ColorMNIST   sample size, cross-fitted nuisance   (M = 2000, 3 seeds)"))

ColorMNIST   sample size, no-split nuisance (default)   (M = 2000, 3 seeds)

n_obs         SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
---------------------------------------------------------------------------------------
5000          0.01058 ± 0.00106    28.363 ± 1.304       0.09782 ± 0.00565    0.02355 ± 0.00669
10000         0.01019 ± 0.00056    30.087 ± 0.825       0.09106 ± 0.00222    0.02343 ± 0.00465
15000         0.01056 ± 0.00084    30.807 ± 0.925       0.08115 ± 0.00083    0.02730 ± 0.00725
20000         0.00939 ± 0.00044    30.618 ± 0.886       0.08016 ± 0.00107    0.02053 ± 0.00357



ColorMNIST   sample size, cross-fitted nuisance   (M = 2000, 3 seeds)

n_obs         SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
---------------------------------------------------------------------------------------
5000          0.01055 ± 0.00137    28.426 ± 1.293       0.09907 ± 0.00730    0.02369 ± 0.00747
10000         0.01055 ± 0.00044    30.116 ± 0.832       0.09067 ± 0.00081    0.02594 ± 0.00492
15000         0.01073 ± 0.00094    30.659 ± 0.956       0.08229 ± 0.00193    0.02976 ± 0.00538
20000         0.00942 ± 0.00016    30.736 ± 0.986       0.08054 ± 0.00132    0.02063 ± 0.00340


Pixel SW2 is flat across the sweep (0.00939–0.01058, no trend beyond the seed spread). FID is
**not** flat — it drifts up 28.363 → 30.087 → 30.807 → 30.618, about 2.3 points — but that drift
belongs to the decoder rather than the sampler: the autoencoder round-trip ceiling moves the same
way over the same sweep (28.642 → 29.979 → 30.591 → 30.576) and the flow stays within 0.3 of it at
every size. `ae_split = none` refits the autoencoder on all `n_obs` logged images at a fixed
3000-iteration budget, so `n_obs` moves the decoder as well as the estimator, and the FID column
cannot be read as sampler quality across this sweep. The one metric that moves for the right reason
is latent SW2: 0.09782 at `n_obs = 5000` down to 0.08016 at 20000, against an oracle floor that
stays near 0.070 at every size — the DR target embedding is still being estimated better with more
logged data even though the decoded images cannot show it. The two nuisance arms track each other
size for size.

Both off-table columns quoted here — `fid_legacy_roundtrip` per `n_obs` and the latent oracle floor —
are read from the same result CSVs these tables are built from; Section 1 tabulates them only at the
default configuration.

## 4. Ablation: number of particles

The `M` sweep at the top sample size (`n_obs = 20000`), no-split nuisance, observed
initialisation.

**Read this section through the count-matched floor.** Sliced Wasserstein is sample-size
biased: a cloud of 250 particles scores better against a reference set purely because it has
fewer points to misplace. The raw floor column does not move with `M` (it is a fixed
2000-vs-2000 oracle draw), so raw values across `M` are not comparable. The
`*_floor_matched` columns re-score the floor at the run's own particle count, and the second
table below reports the ratio to that. A ratio **below 1** means the flow is smoother than an
honest iid sample of the same size — an over-regularisation artefact, not an improvement.

In [7]:
by_m = sweep[(sweep.init == "observed") & (sweep.dr_fit == "nosplit")
             & (sweep.n_obs == 20000)].copy()

print(at.ablation_table(
    by_m, "ColorMNIST", "n_particles_requested", {},
    first_column="M (particles)", label_width=16,
    title="ColorMNIST   raw metric values by particle count   (n_obs = 20000, 3 seeds)",
    note="NOT comparable across rows: every column is sample-size biased. Use the next table."))

ColorMNIST   raw metric values by particle count   (n_obs = 20000, 3 seeds)

M (particles)   SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-----------------------------------------------------------------------------------------
250             0.01737 ± 0.00059    38.487 ± 0.530       0.19151 ± 0.00854    0.03391 ± 0.00258
500             0.01309 ± 0.00042    34.413 ± 0.763       0.12809 ± 0.00542    0.02298 ± 0.00077
1000            0.01090 ± 0.00031    32.115 ± 0.895       0.10185 ± 0.00165    0.02381 ± 0.00181
2000            0.00939 ± 0.00044    30.618 ± 0.886       0.08016 ± 0.00107    0.02053 ± 0.00357

NOT comparable across rows: every column is sample-size biased. Use the next table.

In [8]:
for metric in ["sw2", "latent_sw2", "colour_w2"]:
    by_m[metric + "_ratio"] = by_m[metric + "_flow"] / by_m[metric + "_floor_matched"]

print(at.ablation_table(
    by_m, "ColorMNIST", "n_particles_requested", {},
    metrics=[("SW2 / floor", "sw2_ratio"),
             ("Latent SW2 / floor", "latent_sw2_ratio"),
             ("Color-W2 / floor", "colour_w2_ratio")],
    arrow=False, digits=2, first_column="M (particles)", label_width=16,
    title="ColorMNIST   ratio to the COUNT-MATCHED oracle floor   (n_obs = 20000, 3 seeds)",
    note="1.00 = indistinguishable from an iid oracle sample of the same size; "
         "below 1.00 = over-regularised.\nFID has no count-matched floor column, so it is "
         "absent here; its raw values above are count-biased too."))

ColorMNIST   ratio to the COUNT-MATCHED oracle floor   (n_obs = 20000, 3 seeds)

M (particles)   SW2 / floor          Latent SW2 / floor   Color-W2 / floor
--------------------------------------------------------------------------
250             1.06 ± 0.06          0.97 ± 0.06          1.15 ± 0.10
500             1.04 ± 0.05          0.90 ± 0.11          1.17 ± 0.34
1000            1.20 ± 0.06          0.97 ± 0.09          1.66 ± 0.22
2000            1.46 ± 0.02          1.12 ± 0.06          3.13 ± 0.99

1.00 = indistinguishable from an iid oracle sample of the same size; below 1.00 = over-regularised.
FID has no count-matched floor column, so it is absent here; its raw values above are count-biased too.


The raw table looks like a clean monotone gain (SW2 0.01737 at `M = 250` down to 0.00939 at
`M = 2000`), and it is an artefact. Against the count-matched floor the ordering reverses: the SW2
ratio is flat across the two smallest counts (1.06 and 1.04 — the dip is well inside its own spread)
and then climbs to 1.20 and 1.46, while the latent ratio sits **below 1** at every `M` up to 1000
(0.97 / 0.90 / 0.97) before reaching 1.12 at `M = 2000`. Small-particle clouds are over-regularised —
they beat an iid sample of the same size because they are smoother than one, not because they are
closer to the target. Color-W2 was the intended tie-breaker here, on the assumption that a
one-dimensional statistic is far less count-biased, and this design refutes that assumption: its raw
column is not even monotone (0.03391 / 0.02298 / 0.02381 / 0.02053, rising from `M = 500` to 1000),
and its count-matched floor tightens with `M` faster than either of the others, so its ratio climbs
the fastest of the three (1.15 / 1.17 / 1.66 / 3.13). No column in this section demonstrates a
genuine improvement with `M`. What the two tables together establish is narrower: the raw ordering is
uninterpretable, and only the `M = 2000` cloud is dense enough to be scored honestly, with all three
ratios at or above 1.

## 5. Ablation: initialisation

Where the particles start, at the default configuration otherwise. `N(0, I)` and
`N(mu, Sigma)` are drawn in the raw autoencoder latent coordinates the flow works in;
`N(mu, Sigma)` uses the empirical mean and covariance of the flow's own target support.

In [9]:
by_init = sweep[(sweep.dr_fit == "nosplit") & (sweep.n_obs == 20000)
                & (sweep.n_particles_requested == 2000)]

print(at.ablation_table(
    by_init, "ColorMNIST", "init",
    {"observed": "Observed arm (default)", "moment": "Gaussian N(mu, Sigma)",
     "normal": "Gaussian N(0, I)"},
    order=["observed", "moment", "normal"], first_column="Initialisation", label_width=24,
    title="ColorMNIST   initialisation   (n_obs = 20000, M = 2000, 3 seeds)"))

ColorMNIST   initialisation   (n_obs = 20000, M = 2000, 3 seeds)

Initialisation          SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-------------------------------------------------------------------------------------------------
Observed arm (default)  0.00939 ± 0.00044    30.618 ± 0.886       0.08016 ± 0.00107    0.02053 ± 0.00357
Gaussian N(mu, Sigma)   0.01711 ± 0.00075    50.119 ± 1.000       0.18699 ± 0.01391    0.02412 ± 0.00232
Gaussian N(0, I)        0.02181 ± 0.00056    63.757 ± 1.233       0.16563 ± 0.00596    0.04028 ± 0.00198


Initialisation matters more than anything else in this notebook. Pixel SW2 degrades
0.00939 → 0.01711 → 0.02181 and FID degrades 30.6 → 50.1 → 63.8 from observed to
`N(mu, Sigma)` to `N(0, I)`. Knowing the target's second moment recovers about 40% of the FID
gap (13.6 of the 33.1 points between `N(0, I)` and the observed start), which is the point:
matching moments is not the same as landing on the decoder's image manifold. Two caveats. The
latent column disagrees with the pixel column — `N(mu, Sigma)` is worse in latent SW2 (0.18699
vs 0.16563) while better in every decoded metric — so the flow's own evaluation space does not
rank these inits the way the decoded images do. And all runs share a fixed 3000-iteration
budget, so part of this gap is convergence speed rather than a converged difference.

## 6. Ablation: confounding strength

The `w` sweep (rows tagged `wsweep`), at `n_obs = 20000`, `M = 2000`, observed initialisation.
`w` sets how strongly the logging policy couples colour to the context; `w = 10` is the
default design used everywhere else in this notebook.

**Coverage: only the no-split nuisance arm was run across `w`.** There is no cross-fitted
counterpart at any `w`, so no nuisance comparison is possible in this section.

In [10]:
w_labels = {v: str(int(v)) for v in wsweep.w_conf.unique()}

print(at.ablation_table(
    wsweep, "ColorMNIST", "w_conf", w_labels,
    first_column="w", label_width=12,
    title="ColorMNIST   after the flow, by confounding strength   (no-split, M = 2000, 3 seeds)"))

ColorMNIST   after the flow, by confounding strength   (no-split, M = 2000, 3 seeds)

w           SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-------------------------------------------------------------------------------------
1           0.00736 ± 0.00011    31.178 ± 0.870       0.06853 ± 0.00473    0.01409 ± 0.00278
2           0.00753 ± 0.00019    31.388 ± 0.799       0.06883 ± 0.00469    0.01464 ± 0.00141
3           0.00817 ± 0.00045    31.201 ± 0.919       0.07101 ± 0.00618    0.01679 ± 0.00243
4           0.00795 ± 0.00051    30.757 ± 0.974       0.07198 ± 0.00389    0.01791 ± 0.00211
5           0.00847 ± 0.00030    30.949 ± 0.904       0.07419 ± 0.00569    0.01869 ± 0.00193
6           0.00860 ± 0.00042    30.929 ± 0.288       0.07645 ± 0.00282    0.01933 ± 0.00632
7           0.00841 ± 0.00026    30.377 ± 0.466       0.07852 ± 0.00746    0.01788 ± 0.00290
8           0.00853 ± 0.00045    30.060 ± 0.390       0.07761 ± 0.00373    0.01953 ± 0.0005

In [11]:
print(at.ablation_table(
    wsweep, "ColorMNIST", "w_conf", w_labels,
    metrics=[("SW2 (no flow)", "sw2_initial"), ("Color-W2 (no flow)", "colour_w2_initial"),
             ("ESS on arm", "ess_on_arm"), ("Max weight", "max_weight")],
    digits=[5, 5, 0, 1], arrow=False, first_column="w", label_width=12,
    title="ColorMNIST   what the flow had to correct   (no-split, M = 2000, 3 seeds)",
    note="ESS and max weight are the DR weights on the target arm, out of n_obs = 20000."))

ColorMNIST   what the flow had to correct   (no-split, M = 2000, 3 seeds)

w           SW2 (no flow)        Color-W2 (no flow)   ESS on arm           Max weight
-------------------------------------------------------------------------------------
1           0.01086 ± 0.00036    0.04812 ± 0.00688    9801 ± 29            2.6 ± 0.1
2           0.01800 ± 0.00055    0.09606 ± 0.00086    9208 ± 50            3.7 ± 0.1
3           0.02329 ± 0.00108    0.12725 ± 0.00574    8283 ± 62            5.4 ± 0.1
4           0.02746 ± 0.00122    0.15597 ± 0.00750    7159 ± 73            8.2 ± 0.2
5           0.03201 ± 0.00054    0.17995 ± 0.00392    5854 ± 150           13.1 ± 0.6
6           0.03452 ± 0.00050    0.19816 ± 0.00711    4620 ± 144           20.9 ± 1.0
7           0.03804 ± 0.00047    0.21804 ± 0.00237    3488 ± 190           33.8 ± 2.4
8           0.04081 ± 0.00126    0.23358 ± 0.00598    2576 ± 102           53.2 ± 3.1
9           0.04224 ± 0.00049    0.24196 ± 0.00144    1874 ± 83      

The second table is the size of the problem: as `w` goes 1 → 10 the logged arm drifts from
SW2 0.01086 to 0.04412 (a 4.1x miscalibration) and Color-W2 from 0.04812 to 0.25122, while the
effective sample size of the DR weights collapses 9801 → 1383 and the largest single weight grows
2.6 → 126.4. The first table is the flow's answer: SW2 after the flow moves only 0.00736 → 0.00938
and FID is flat at 30.1–31.4 across the whole range. The correction degrades gracefully and is
driven by the weights, not by the sampler.

One bookkeeping note: `w = 10` here is an independent re-run of the default configuration and
scores SW2 0.00938 against the 0.00939 of Section 1, with FID 30.69 against 30.62. The DR weights
are bit-identical between the two (the ESS and max-weight columns match exactly, row for row), so
the residual gap is run-to-run autoencoder training noise — an order of magnitude below the seed
spread on both columns.

## Summary

- **We lead the published baselines on the two headline metrics.** SW2 0.00939 ± 0.00044
  against DecFM-Plugin's 0.01137 ± 0.00078, and FID 30.618 ± 0.886 against 32.005 / 33.650 /
  33.824. Latent SW2 (0.08016 ± 0.00107) and Color-W2 (0.02053 ± 0.00357) sit inside the range
  the three baselines span.
- **The ColorMNIST FID column measures the decoder, not the sampler.** The oracle sample
  passed through the same 28-pixel autoencoder scores `fid_legacy_roundtrip` = 30.576 ± 0.761,
  and our flow scores 30.618 ± 0.886 — a gap of 0.04 against a seed spread of 0.9. The raw
  oracle floor is 3.343 ± 0.102. Improving FID on this benchmark means a better decoder.
- **Cross-fitting is not needed here.** 0.00942 ± 0.00016 vs 0.00939 ± 0.00044 on SW2, and
  every other column likewise inside one standard deviation.
- **More logged data helps the embedding, not the image.** Latent SW2 falls 0.09782 → 0.08016
  from `n_obs = 5000` to 20000 while pixel SW2 stays flat at 0.00939–0.01058. FID drifts *up*
  over the same sweep (28.363 → 30.618), but the round-trip ceiling drifts with it (28.642 →
  30.576): `ae_split = none` refits the autoencoder on all `n_obs` images, so that column
  tracks the decoder and not the sampler at every size, not only at the default.
- **The particle-count gain is an artefact of sample-size bias.** Against the count-matched
  floor every ratio is *worse* at `M = 2000` than at `M = 250`: SW2 1.06 → 1.46 (dipping to
  1.04 at `M = 500`, inside its own spread) and Color-W2 1.15 → 3.13. The latent ratio is below
  1 at every `M` up to 1000 (0.97 / 0.90 / 0.97) — below an honest iid sample of the same size —
  before reaching 1.12 at `M = 2000`. Color-W2 does not rescue the raw reading either: its raw
  column is non-monotone (0.03391 / 0.02298 / 0.02381 / 0.02053) and its count-matched floor
  tightens fastest of the three, so no column certifies a genuine gain from more particles.
- **Initialisation is the largest effect measured.** Observed 0.00939 / FID 30.6 against
  `N(mu, Sigma)` 0.01711 / 50.1 and `N(0, I)` 0.02181 / 63.8, at a fixed 3000-iteration budget.
  Knowing the target's second moment recovers 41% of the FID gap (13.6 of the 33.1 points).
  Latent SW2 ranks the two Gaussian inits the other way round (0.18699 vs 0.16563), so it
  cannot certify decoded sample quality here.
- **The correction holds as confounding grows.** Over `w = 1…10` the uncorrected arm degrades
  0.01086 → 0.04412 in SW2 and the weight ESS falls 9801 → 1383, while the flow's SW2 moves
  only 0.00736 → 0.00938 and its FID stays flat at 30.1–31.4. Only the no-split nuisance arm
  was run across `w`, so this section has no cross-fitted counterpart.